In [3]:
!pip install einops
!pip install torchsummary
!pip install torchvision

  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
Using cached einops-0.8.2-py3-none-any.whl (65 kB)
  Using cached torchvision-0.25.0-cp313-cp313-win_amd64.whl.metadata (5.4 kB)
  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
Using cached torchvision-0.25.0-cp313-cp313-win_amd64.whl (4.3 MB)
Using cached torch-2.10.0-cp313-cp313-win_amd64.whl (113.8 MB)

   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2

In [4]:
import einops
from tqdm.notebook import tqdm

from torchsummary import summary

import time

import torch
from torch import nn
# for dataset and dataloader
import torchvision
#optimizier
import torch.optim as optim
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, RandomHorizontalFlip, RandomCrop

In [14]:
# using GPU if available
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

patch_size = 16         # Patch size (P) = 16
latent_size = 768       # Latent vector (D). ViT-Base uses 768
n_channels = 3          # Number of channels for input images
num_heads = 12          # ViT-Base uses 12 heads
num_encoders = 12       # ViT-Base uses 12 encoder layers
dropout = 0.1           # Dropout = 0.1 is used with ViT-Base & ImageNet-21k
num_classes = 10        # Number of classes in CIFAR10 dataset
size = 224              # Size used for training = 224

epochs = 10             # Number of epochs
base_lr = 10e-3         # Base LR
weight_decay = 0.03     # Weight decay for ViT-Base (on ImageNet-21k)
batch_size = 1

cpu


Implementation of input layer projection

In [ ]:
class InputEmbedding(nn.Module):
    def __init__(self, patch_size=patch_size,n_channels=n_channels, device=device, latent_size=latent_size, batch_size= batch_size):
        super(InputEmbedding,self).__init__()
        self.patch_size = patch_size
        self.latent_size = latent_size
        self.n_channels = n_channels
        self.device = device
        self.batch_size = batch_size
        self.input_size = self.patch_size * self.patch_size * self.n_channels
        # Linear Projection
        self.linearProjection = nn.Linear(self.input_size, self.latent_size)
        # Class token
        self.class_token = nn.Parameter(torch.randn(self.batch_size, 1, self.latent_size)).to(self.device)
        #Postional embedding
        self.pos_embedding = nn.Parameter(torch.randn(self.batch_size, 1, self.latent_size)).to(self.device)
    def forward(self, input_data):
        input_data = input_data.to(self.device)

        #Patchify input image
        patches = einops.rearrange(
            input_data, 'b c (h h1) (w w1) -> b (h w) (h1 w1 c)', h1=self.patch_size, w1=self.patch_size)
        
        # print(input_data.size())
        # print(patches.size())

        linear_projection = self.linearProjection(patches)
        b, n , _ = linear_projection.size()

        linear_projection = torch.cat((self.class_token, linear_projection), dim=1)
        pos_emb = einops.repeat(self.pos_embedding, 'b 1 d -> b m d', m=n)
        linear_projection += pos_emb
        return linear_projection

Encoder Block

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, latent_size=latent_size, num_heads= num_heads, device = device, dropout=dropout):
        super(EncoderBlock).__init__()
        self.latent_size=latent_size
        self.num_heads=num_heads
        self.device=device
        self.dropout=dropout

        # Normalization Layer 
        self.norm = nn.LayerNorm(self.latent_size)

        self.multihead = nn.MultiheadAttention(
            self.latent_size, self.num_heads, dropout=self.dropout
        )    
        self.enc_MLP = nn.Sequential(
            nn.Linear(latent_size, latent_size*4),
            nn.GELU(),
            nn.Dropout(self.dropout)
            nn.Linear(latent_size*4, latent_size),
            nn.Dropout(self.dropout)
        )
    def forward(self, embedded_patches):
        firstnorm_out = self.norm(embedded_patches)
        attention_out = self.multihead(firstnorm_out,firstnorm_out,firstnorm_out)[0]

        